In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Environment Setup
Install Python 3.10, set it as default, reinstall pip, and verify the installation along with GPU availability.

In [ ]:
# 1. Install Python 3.10
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-distutils -y

# 2. Add Python 3.10 to the update-alternatives list
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1

# 3. Configure the default (you'll need to input the selection number if prompted)
# In Colab, you can often skip the manual prompt by setting priority,
# but this command ensures the link is created.
!sudo update-alternatives --set python3 /usr/bin/python3.10

# 4. Reinstall pip for the new version
!curl https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!python3 get-pip.py --force-reinstall

# 5. Check the version
!python --version

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,601 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,245 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,245 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [61.6 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-up

In [ ]:
!python --version

import sys
print(sys.version)
print(sys.executable)

Python 3.10.12
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
/usr/bin/python3


In [ ]:
# Verify GPU is available
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: Tesla T4


## Clone Repository & Install Dependencies
Download the XTTSv2 fine‑tuning code, remove conflicting blinker packages, and install all required libraries.

In [ ]:
import os

REPO_DIR = "/content/XTTSv2-Finetuning-for-New-Languages"

if not os.path.exists(REPO_DIR):
    # !git clone https://github.com/nguyenhoanganh2002/XTTSv2-Finetuning-for-New-Languages.git {REPO_DIR}
    !git clone https://github.com/Fabzamm/XTTSv2-Finetuning-for-New-Languages.git {REPO_DIR}

%cd {REPO_DIR}

Cloning into '/content/XTTSv2-Finetuning-for-New-Languages'...
remote: Enumerating objects: 726, done.
remote: Counting objects: 100% (316/316), done.
remote: Compressing objects: 100% (218/218), done.
remote: Total 726 (delta 137), reused 98 (delta 98), pack-reused 410 (from 3)
Receiving objects: 100% (726/726), 2.12 MiB | 19.21 MiB/s, done.
Resolving deltas: 100% (213/213), done.
/content/XTTSv2-Finetuning-for-New-Languages


In [ ]:
# Force-remove pre-installed 'blinker' to avoid version conflicts in Colab
!find /usr/lib/python3 -name "blinker*" -exec rm -rf {} + 2>/dev/null
!find /usr/local/lib/python3.10 -name "blinker*" -exec rm -rf {} + 2>/dev/null

In [ ]:
!pip install torchcodec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 20.6 MB/s  0:00:00


In [ ]:
# Took around 5-6 minutes

%cd /content/XTTSv2-Finetuning-for-New-Languages
!pip install -r requirements.txt

/content/XTTSv2-Finetuning-for-New-Languages
Ignoring numpy: markers 'python_version > "3.10"' don't match your environment
Ignoring numba: markers 'python_version < "3.9"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 61.1 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 114.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing met

## Mount Google Drive & Configure Paths
Mount Drive to access the dataset and set all necessary paths for training.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import pandas as pd
import os

# Language code for your target language
LANGUAGE = "mt"  # e.g. "mt" for Maltese

# ── Dataset paths ────────────────────────────────────────────────────────────
MASRI_DIR = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/Train+Dev/MASRI"
CV_DIR    = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/Train+Dev/CV"

MASRI_TRAIN_CSV = os.path.join(MASRI_DIR, "metadata_train.csv")
MASRI_EVAL_CSV  = os.path.join(MASRI_DIR, "metadata_eval.csv")
CV_TRAIN_CSV    = os.path.join(CV_DIR, "metadata_train.csv")
CV_EVAL_CSV     = os.path.join(CV_DIR, "metadata_eval.csv")

# ── Merge both datasets into combined CSVs ───────────────────────────────────
COMBINED_DIR = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/Train+Dev/Combined"

COMBINED_TRAIN_CSV = os.path.join(COMBINED_DIR, "metadata_train.csv")
COMBINED_EVAL_CSV  = os.path.join(COMBINED_DIR, "metadata_eval.csv")

print(f"Combined train rows: {len(pd.read_csv(COMBINED_TRAIN_CSV, sep='|'))}")
print(f"Combined eval rows:  {len(pd.read_csv(COMBINED_EVAL_CSV, sep='|'))}")

# # ── Sanity check: verify first row of each source resolves ───────────────────
# df_check = pd.read_csv(COMBINED_TRAIN_CSV, sep="|")
# print(f"\nSample path: {df_check.iloc[6000]['audio_file']}")
# print(f"Exists: {os.path.exists(df_check.iloc[6000]['audio_file'])}")

# Where checkpoints will be saved (Drive keeps them across sessions)
CHECKPOINT_DIR = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints"

# Extended vocab size for the new language's BPE tokens
EXTENDED_VOCAB_SIZE = 1000   # 500  # 750 # 1000

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Checkpoints → {CHECKPOINT_DIR}")

Combined train rows: 8886
Combined eval rows:  1883
Checkpoints → /content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints


## Prepare Base Model & Extend Vocabulary
Download the pretrained XTTSv2 checkpoint and adapt its tokenizer for the new language.


In [ ]:
# !python download_checkpoint.py --output_path {CHECKPOINT_DIR} # Don't run again

In [ ]:
# # # Don't run again

# # # !python extend_vocab_config.py \
# # #     --output_path={CHECKPOINT_DIR} \
# # #     --metadata_path={TRAIN_CSV} \
# # #     --language={LANGUAGE} \
# # #     --extended_vocab_size={EXTENDED_VOCAB_SIZE}

# # !python extend_vocab_config.py \
# #     --output_path={CHECKPOINT_DIR} \
# #     --metadata_path={COMBINED_TRAIN_CSV} \
# #     --language={LANGUAGE} \
# #     --extended_vocab_size={EXTENDED_VOCAB_SIZE}

# # from google.colab import userdata
# # from huggingface_hub import login

# # login(token=userdata.get("HF_TOKEN"))

# !python extend_vocab_config.py \
#     --output_path={CHECKPOINT_DIR} \
#     --metadata_path={COMBINED_TRAIN_CSV} \
#     --language={LANGUAGE} \
#     --extended_vocab_size={EXTENDED_VOCAB_SIZE} \
#     --use_korpus
#     # --korpus_max_samples=3414250 # 3414250 = 10%

## Train DVAE Component
Fine‑tune the duration variational autoencoder on the new language.

In [ ]:
# # Took 17 minutes (epoch = 1, batch = 128, lr = 5e-6)
# # Took a little over 1 hr (epoch = 5, batch = 128, lr = 5e-6) (With MASRI)
# # For about 1 epoch takes 1 hr (epoch = 5, batch = 128, lr = 5e-6) (With MASRI+Common_voice --> 81 steps (10416 files)) (01:33:14 / 01:12:50)
# # 6089.329 s (5 epochs)

# # ── DVAE training hyper-parameters ───────────────────────────────────────────
# DVAE_EPOCHS     = 10 # Recommended num_epochs=5
# DVAE_BATCH_SIZE = 128   # reduce if you run out of VRAM (Originally 512)
# DVAE_LR         = 5e-6
# # ─────────────────────────────────────────────────────────────────────────────

# !python train_dvae_xtts.py \
#     --output_path={CHECKPOINT_DIR} \
#     --train_csv_path={COMBINED_TRAIN_CSV} \
#     --eval_csv_path={COMBINED_EVAL_CSV} \
#     --language={LANGUAGE} \
#     --num_epochs={DVAE_EPOCHS} \
#     --batch_size={DVAE_BATCH_SIZE} \
#     --lr={DVAE_LR}

## Train GPT Component
Patch torch.load for compatibility and fine‑tune the GPT decoder.

In [ ]:
# import psutil
# print(f"Available RAM: {psutil.virtual_memory().available / 1e9:.1f} GB")

In [ ]:
# Epochs = 5 took about 2 hrs (Epochs=5, batch=3, grad accum=4, max_text=400, max_audio=330750, weight_decay=1e-2, lr=5e-6, save_step=1300) [MASRI only]
# Epochs = 1 took [00:53:49] (Epochs=1, batch=3, grad accum=4, max_text=400, max_audio=330750, weight_decay=1e-2, lr=5e-6, save_step=3471) [MASRI + Common Voice (3472 steps)]

# 2 Epochs took 3 hours, 36 minutes, and 27 seconds

# Another 2 epochs (6 hours, 29 minutes, and 3 seconds) [For Epochs 9 & 10]
# For Epochs 1 and 2 (With DVAE): 1 hour, 32 minutes, and 49 seconds

#16986.72 s

# For GPT 7,8 (Korpus) --> 15675.144 s

# ── GPT training hyper-parameters ────────────────────────────────────────────
GPT_EPOCHS              = 3 # Starting point 10 --> Recommended 100+
GPT_BATCH_SIZE          = 2 # was 3
GPT_GRAD_ACCUM          = 16 # was 4
GPT_MAX_TEXT_LEN        = 400
GPT_MAX_AUDIO_LEN       = 330750    # ~15 seconds @ 22050 Hz
GPT_WEIGHT_DECAY        = 1e-2
GPT_LR                  = 5e-6
GPT_SAVE_STEP           = 4443 #4442 # (was 500) [3472]
GPT_SAVE_N_CHECKPOINTS  = 100
# ─────────────────────────────────────────────────────────────────────────────

METADATA_ARG_MASRI    = f"{MASRI_TRAIN_CSV},{MASRI_EVAL_CSV},{LANGUAGE}"
METADATA_ARG_CV       = f"{CV_TRAIN_CSV},{CV_EVAL_CSV},{LANGUAGE}"

RESTORE_CHECKPOINT = "/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_adding_Korpus_Full/9_Epochs_GPT_XTTS_FT-May-05-2026_06+04AM-7048429/checkpoint_39987.pth"
# RESTORE_CHECKPOINT = None
restore_arg = f"--restore_path {RESTORE_CHECKPOINT}" if RESTORE_CHECKPOINT else ""

!python train_gpt_xtts.py \
    --output_path {CHECKPOINT_DIR} \
    --metadatas "{METADATA_ARG_MASRI}" "{METADATA_ARG_CV}" \
    --num_epochs {GPT_EPOCHS} \
    --batch_size {GPT_BATCH_SIZE} \
    --grad_acumm {GPT_GRAD_ACCUM} \
    --max_text_length {GPT_MAX_TEXT_LEN} \
    --max_audio_length {GPT_MAX_AUDIO_LEN} \
    --weight_decay {GPT_WEIGHT_DECAY} \
    --lr {GPT_LR} \
    --save_step {GPT_SAVE_STEP} \
    --save_n_checkpoints {GPT_SAVE_N_CHECKPOINTS} \
    {restore_arg}

/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/Train+Dev/MASRI/metadata_train.csv /content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/Train+Dev/MASRI/metadata_eval.csv mt
/content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/Train+Dev/CV/metadata_train.csv /content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/Train+Dev/CV/metadata_eval.csv mt
 > Loading checkpoint with 620 additional tokens.
>> DVAE weights restored from: /content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/XTTS_v2.0_original_model_files/dvae.pth
 | > Found 4963 files in /content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/Train+Dev/MASRI
 | > Found 3923 files in /content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/Train+Dev/CV
 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: False
 | > Precision: float32
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 2
 | > Num. of Torch Threads: 1
 | > Torch seed: 54321
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch

In [ ]:
# # Load TensorBoard extension
# %load_ext tensorboard

# # Start TensorBoard pointing to your checkpoints folder
# %tensorboard --logdir /content/drive/Shareddrives/UMSpeech/TTS_Work_Fabio/checkpoints/Checkpoints_for_vocab_750